In [1]:
import json
import re
import os
from bs4 import BeautifulSoup

def clean_wiki_text(text):
    """Làm sạch văn bản: xóa chú thích [1], [ghi chú], ký tự lạ và khoảng trắng thừa."""
    if not text: return ""
    # Xóa tham chiếu chú thích dạng [1], [2], [a], [ghi chú 1]
    text = re.sub(r'\[[^\]]*\]', '', text)
    # Thay thế các ký tự trắng đặc biệt và xuống dòng thừa
    text = text.replace('\xa0', ' ').replace('\n', ' ')
    # Xóa khoảng trắng thừa giữa các từ
    return re.sub(r'\s+', ' ', text).strip()

def extract_comprehensive_text(input_file, output_json):
    if not os.path.exists(input_file):
        print(f"Lỗi: Không tìm thấy file {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    # 1. Lấy Tiêu đề bài viết
    title_tag = soup.find('h1', id='firstHeading')
    title = title_tag.get_text(strip=True) if title_tag else "Unknown Title"

    # 2. Trích xuất Infobox (Dữ liệu có cấu trúc bên phải)
    infobox = {}
    info_table = soup.find('table', class_='infobox')
    if info_table:
        for row in info_table.find_all('tr'):
            label = row.find(['th', 'td'], class_='infobox-label') or row.find('th')
            value = row.find(['td'], class_='infobox-data') or row.find('td')
            if label and value and label != value:
                key = clean_wiki_text(label.get_text())
                val = clean_wiki_text(value.get_text())
                if key: infobox[key] = val

    # 3. Trích xuất toàn bộ Plain Text phân theo mục (Sections)
    content_body = soup.find('div', class_='mw-parser-output')
    sections = {}
    
    if content_body:
        # Lấy tóm tắt mở đầu (trước khi gặp tiêu đề đầu tiên hoặc mục lục)
        intro_text = []
        for elem in content_body.find_all(['p', 'ul'], recursive=False):
            txt = clean_wiki_text(elem.get_text())
            if txt and len(txt) > 20: # Tránh lấy các dòng vụn vặt
                intro_text.append(txt)
        sections["Tóm tắt mở đầu"] = " ".join(intro_text)

        # Duyệt qua các tiêu đề h2, h3 để lấy nội dung chi tiết
        current_header = None
        for elem in content_body.find_all(['h2', 'h3', 'p', 'ul']):
            if elem.name in ['h2', 'h3']:
                # Lấy tên tiêu đề, bỏ các chữ điều hướng như [sửa], [sửa mã nguồn]
                header_text = clean_wiki_text(elem.get_text().replace('sửa', '').replace('mã nguồn', ''))
                # Loại bỏ các mục phụ lục không mang thông tin chính về nhân vật
                if header_text in ["Xem thêm", "Ghi chú", "Tham khảo", "Sách tham khảo", "Liên kết ngoài"]:
                    current_header = None
                else:
                    current_header = header_text
            elif current_header and elem.name in ['p', 'ul']:
                txt = clean_text = clean_wiki_text(elem.get_text())
                if txt and len(txt) > 10:
                    sections.setdefault(current_header, []).append(txt)

        # Gộp danh sách text trong mỗi section thành chuỗi hoàn chỉnh
        for k in sections:
            if isinstance(sections[k], list):
                sections[k] = " ".join(sections[k])

    # 4. Trích xuất Categories (Để phân loại dữ liệu)
    categories = []
    cat_links = soup.find('div', id='mw-normal-catlinks')
    if cat_links:
        categories = [clean_wiki_text(li.get_text()) for li in cat_links.find_all('li')]

    # Tổng hợp dữ liệu cuối cùng
    final_data = {
        "title": title,
        "infobox": infobox,
        "content_by_sections": sections,
        "categories": categories
    }

    # Lưu ra file JSON
    with open(output_json, 'w', encoding='utf-8') as out_f:
        json.dump(final_data, out_f, ensure_ascii=False, indent=4)
    
    print(f"--- THÀNH CÔNG ---")
    print(f"Đã trích xuất Plain Text từ '{title}' vào file '{output_json}'")

# --- THỰC THI ---
if __name__ == "__main__":
    # Thay 'first_trial.txt' bằng file của bạn
    extract_comprehensive_text(r'D:\MScThi\12_02_2026\first_trial.txt', 'wiki_plain_text.json')

--- THÀNH CÔNG ---
Đã trích xuất Plain Text từ 'Niên biểu lịch sử Việt Nam' vào file 'wiki_plain_text.json'


**EXTRACT WENHUA**

In [2]:
import json
import re
import os
import requests
import time
from urllib.parse import quote

class WikiCulturalAgent:
    def __init__(self, input_json, storage_folder="van_hoa_storage"):
        self.input_json = input_json
        self.storage_folder = storage_folder
        self.headers = {
            'User-Agent': 'CulturalResearchBot/1.0 (Contact: your_email@example.com)'
        }
        if not os.path.exists(self.storage_folder):
            os.makedirs(self.storage_folder)

    def extract_keywords_from_json(self):
        """Đọc file JSON và trích xuất tất cả cụm từ 'Văn hóa...'"""
        if not os.path.exists(self.input_json):
            print(f"Lỗi: Không tìm thấy file {self.input_json}")
            return []

        with open(self.input_json, 'r', encoding='utf-8') as f:
            data = json.load(f)

        keywords = set()
        # Regex: Tìm 'Văn hóa' theo sau bởi các từ viết hoa (tên riêng của nền văn hóa)
        # Hỗ trợ tiếng Việt có dấu
        pattern = r"Văn hóa\s+[A-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠƯ][\w\s]*?(?=\s+\d|\s+Văn|[,.;]|$)"

        # Duyệt qua tất cả các section trong nội dung JSON
        content_dict = data.get("content_by_sections", {})
        for text in content_dict.values():
            matches = re.findall(pattern, text)
            for m in matches:
                name = m.strip()
                # Loại bỏ các từ thừa ở cuối nếu có (như dấu gạch ngang hoặc khoảng trắng)
                name = re.sub(r'\s+[-–]\s*$', '', name)
                if len(name) > 8: # Tránh lấy mỗi chữ "Văn hóa" trống
                    keywords.add(name)
        
        return sorted(list(keywords))

    def fetch_and_save(self, keyword):
        """Tải HTML từ Wikipedia và lưu thành file txt"""
        safe_filename = keyword.replace(" ", "_") + ".txt"
        file_path = os.path.join(self.storage_folder, safe_filename)

        if os.path.exists(file_path):
            return f"SKIP: {keyword} đã tồn tại."

        url = f"https://vi.wikipedia.org/wiki/{quote(keyword.replace(' ', '_'))}"
        try:
            response = requests.get(url, headers=self.headers, timeout=15)
            if response.status_code == 200:
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(response.text)
                return f"SUCCESS: Đã lưu {keyword}"
            else:
                return f"FAILED: {keyword} (Mã lỗi {response.status_code})"
        except Exception as e:
            return f"ERROR: {keyword} - {str(e)}"

    def run(self):
        print(f"Đang đọc dữ liệu từ {self.input_json}...")
        targets = self.extract_keywords_from_json()
        print(f"Tìm thấy {len(targets)} nền văn hóa cần tra cứu.")
        
        for name in targets:
            log = self.fetch_and_save(name)
            print(log)
            time.sleep(1) # Nghỉ 1 giây để không bị Wiki chặn

# --- THỰC THI ---
if __name__ == "__main__":
    # Đảm bảo file nien_bieu_vn.json của bạn nằm cùng thư mục
    agent = WikiCulturalAgent(input_json=r'D:\MScThi\12_02_2026\NienBieuLichSuVietNam\wiki_plain_text.json')
    agent.run()

Đang đọc dữ liệu từ D:\MScThi\12_02_2026\NienBieuLichSuVietNam\wiki_plain_text.json...
Tìm thấy 18 nền văn hóa cần tra cứu.
SUCCESS: Đã lưu Văn hóa Bắc Sơn
SUCCESS: Đã lưu Văn hóa Cái Bèo
SUCCESS: Đã lưu Văn hóa Gò Mun
SUCCESS: Đã lưu Văn hóa Hòa Bình
SUCCESS: Đã lưu Văn hóa Hạ Long
SUCCESS: Đã lưu Văn hóa Ngườm
SUCCESS: Đã lưu Văn hóa Phùng Nguyên
FAILED: Văn hóa Quỳnh (Mã lỗi 404)
SUCCESS: Đã lưu Văn hóa Sa Huỳnh
SUCCESS: Đã lưu Văn hóa Soi Nhụ
SUCCESS: Đã lưu Văn hóa Sơn Vi
FAILED: Văn hóa Tiền Sa Huỳnh (Mã lỗi 404)
SUCCESS: Đã lưu Văn hóa Tràng An
SUCCESS: Đã lưu Văn hóa Óc Eo
SUCCESS: Đã lưu Văn hóa Đa Bút
SUCCESS: Đã lưu Văn hóa Đông Sơn
SUCCESS: Đã lưu Văn hóa Đồng Nai
SUCCESS: Đã lưu Văn hóa Đồng Đậu


**EXTRACT EACH WENHUA**

In [7]:
import os
import json
import re
from bs4 import BeautifulSoup

class BulkWikiProcessor:
    def __init__(self, input_folder="van_hoa_storage", output_file="database_lich_su_tong_hop.json"):
        self.input_folder = input_folder
        self.output_file = output_file
        self.processed_titles = set() # Bộ nhớ để khử trùng lặp

    def clean_text(self, text):
        """Làm sạch văn bản: xóa chú thích [1], xóa ký tự lạ, chuẩn hóa khoảng trắng."""
        if not text: return ""
        # Xóa tham chiếu chú thích dạng [1], [2], [ghi chú...]
        text = re.sub(r'\[[^\]]*\]', '', text)
        # Thay thế các ký tự trắng đặc biệt và ngắt dòng
        text = text.replace('\xa0', ' ').replace('\n', ' ')
        # Xóa khoảng trắng thừa giữa các từ
        return re.sub(r'\s+', ' ', text).strip()

    def parse_wiki_file(self, file_path):
        """Hàm trích xuất nội dung chi tiết từ 1 file HTML Wikipedia."""
        with open(file_path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')

        # 1. Tên thực thể (Tiêu đề h1)
        title_tag = soup.find('h1', id='firstHeading')
        title = title_tag.get_text(strip=True) if title_tag else "Unknown"

        # Khử trùng: Nếu tiêu đề này đã được xử lý từ file khác, bỏ qua
        norm_title = title.lower().strip()
        if norm_title in self.processed_titles or title == "Unknown":
            return None
        self.processed_titles.add(norm_title)

        # 2. Vùng nội dung chính
        content_body = soup.find('div', class_='mw-parser-output')
        data = {
            "title": title,
            "summary": "",
            "infobox_sidebar": {},
            "sections": {},
            "categories": []
        }

        if content_body:
            # --- Trích xuất Infobox hoặc Sidebar (Bảng thông số bên phải) ---
            info_table = content_body.find('table', class_=re.compile(r'infobox|sidebar'))
            if info_table:
                for row in info_table.find_all('tr'):
                    th = row.find(['th', 'td'], class_=re.compile(r'label|heading')) or row.find('th')
                    td = row.find(['td'], class_=re.compile(r'data')) or row.find('td')
                    if th and td and th != td:
                        k = self.clean_text(th.get_text())
                        v = self.clean_text(td.get_text())
                        if k: data["infobox_sidebar"][k] = v

            # --- Trích xuất Tóm tắt & Nội dung theo từng mục ---
            current_header = "Giới thiệu"
            # Duyệt qua các thẻ con trực tiếp để đảm bảo thứ tự
            for elem in content_body.find_all(['h2', 'h3', 'p', 'ul', 'ol'], recursive=False):
                if elem.name in ['h2', 'h3']:
                    h_text = self.clean_text(elem.get_text().replace('sửa', '').replace('mã nguồn', ''))
                    # Loại bỏ các mục lục rác ở cuối trang
                    if h_text in ["Tham khảo", "Liên kết ngoài", "Ghi chú", "Xem thêm", "Nguồn", "Sách tham khảo chính"]:
                        current_header = None
                    else:
                        current_header = h_text
                elif current_header:
                    txt = self.clean_text(elem.get_text())
                    if txt and len(txt) > 15: # Tránh các dòng quá ngắn/rác
                        if current_header == "Giới thiệu":
                            data["summary"] += txt + " "
                        else:
                            data["sections"].setdefault(current_header, []).append(txt)

            # Gộp mảng text trong sections thành chuỗi hoàn chỉnh
            data["summary"] = data["summary"].strip()
            for k in data["sections"]:
                data["sections"][k] = " ".join(data["sections"][k])

        # 3. Trích xuất Thể loại (Categories) ở cuối trang
        cat_div = soup.find('div', id='mw-normal-catlinks')
        if cat_div:
            data["categories"] = [self.clean_text(li.get_text()) for li in cat_div.find_all('li')]

        return data

    def run(self):
        """Quét toàn bộ folder và gộp vào file JSON duy nhất."""
        all_data = []
        if not os.path.exists(self.input_folder):
            print(f"Lỗi: Thư mục '{self.input_folder}' không tồn tại.")
            return

        # Lấy danh sách file .txt
        files = [f for f in os.listdir(self.input_folder) if f.endswith('.txt')]
        print(f"Bắt đầu xử lý {len(files)} tệp tin...")

        for idx, filename in enumerate(files):
            path = os.path.join(self.input_folder, filename)
            try:
                result = self.parse_wiki_file(path)
                if result:
                    all_data.append(result)
                    print(f"[{idx+1}/{len(files)}] √ Đã xử lý: {result['title']}")
                else:
                    print(f"[{idx+1}/{len(files)}] - Bỏ qua (Trùng lặp hoặc Unknown): {filename}")
            except Exception as e:
                print(f"[{idx+1}/{len(files)}] x Lỗi tại {filename}: {str(e)}")

        # Lưu kết quả
        with open(self.output_file, 'w', encoding='utf-8') as out_f:
            json.dump(all_data, out_f, ensure_ascii=False, indent=4)
        
        print(f"\n--- HOÀN TẤT ---")
        print(f"Tổng số thực thể duy nhất đã trích xuất: {len(all_data)}")
        print(f"Kết quả được lưu tại: {os.path.abspath(self.output_file)}")

# --- THỰC THI ---
if __name__ == "__main__":
    # Thay đổi 'van_hoa_storage' thành tên folder chứa các file .txt của bạn
    processor = BulkWikiProcessor(input_folder="van_hoa_storage", output_file="merged_data.json")
    processor.run()

Bắt đầu xử lý 16 tệp tin...
[1/16] √ Đã xử lý: Văn hóa Bắc Sơn
[2/16] √ Đã xử lý: Văn hóa Cái Bèo
[3/16] √ Đã xử lý: Văn hóa Gò Mun
[4/16] √ Đã xử lý: Văn hóa Hòa Bình
[5/16] √ Đã xử lý: Văn hóa Hạ Long
[6/16] √ Đã xử lý: Kỹ nghệ Ngườm
[7/16] √ Đã xử lý: Văn hóa Phùng Nguyên
[8/16] √ Đã xử lý: Văn hóa Sa Huỳnh
[9/16] √ Đã xử lý: Văn hóa Soi Nhụ
[10/16] √ Đã xử lý: Văn hóa Sơn Vi
[11/16] √ Đã xử lý: Văn hóa Tràng An
[12/16] √ Đã xử lý: Văn hóa Óc Eo
[13/16] √ Đã xử lý: Văn hóa Đa Bút
[14/16] √ Đã xử lý: Văn hóa Đông Sơn
[15/16] √ Đã xử lý: Văn hóa Đồng Nai
[16/16] √ Đã xử lý: Văn hóa Đồng Đậu

--- HOÀN TẤT ---
Tổng số thực thể duy nhất đã trích xuất: 16
Kết quả được lưu tại: d:\MScThi\12_02_2026\NienBieuLichSuVietNam\merged_data.json
